In [1]:
import logging
import matplotlib.pyplot as plt
import numpy as np
import os
import sys

# Add the parent directory to the path to import from rainbow
sys.path.append(os.path.dirname(os.path.abspath('')))

import rainbow.math.vector3 as V3
import rainbow.math.quaternion as Q
import rainbow.simulators.prox_rigid_bodies.api as API
import rainbow.geometry.surface_mesh as MESH
import rainbow.simulators.prox_rigid_bodies.types as TYPES
import rainbow.procedural.gears as GEAR

from gear_app import GearApp

In [ ]:
logging.basicConfig(level=logging.INFO)

sdf_resolution = 128

engine = API.create_engine()
engine.params.time_step = 0.01
engine.params.sdf_min_cells = sdf_resolution
engine.params.sdf_max_cells = sdf_resolution
engine.params.resolution = sdf_resolution
app = GearApp(engine, f'PlanetaryGear_{sdf_resolution}')

face_width = 10
factory = GEAR.generators.GearFactory()

m = 1
z_sun = 31
z_planet = 17
z_ring = z_sun + 2 * z_planet
planetary_spec = GEAR.types.PlanetaryGearSpec(m, z_sun, z_planet, z_ring, N_planet=3)
planetary_gear = factory.create_planetary_gear(planetary_spec, face_width / 2, subdivisions=3)

sun_gear = planetary_gear.sun_gear
planet_gears = planetary_gear.planet_gears
ring_gear = planetary_gear.ring_gear

sun_name = API.add_object(engine, 'sun_gear', sun_gear.mesh, sun_gear.position, sun_gear.orientation)

planet_names = []
for i, gear in enumerate(planet_gears):
    planet_name = API.add_object(engine, f'planet_gear_{i}', gear.mesh, gear.position, gear.orientation)
    planet_names.append(planet_name)

ring_name = API.add_object(engine, 'ring_gear', ring_gear.mesh, ring_gear.position, ring_gear.orientation, body_type='fixed')

V, T = MESH.create_cylinder(ring_gear.spec.rd * 0.95, face_width / 2, 18)
flywheel_mesh = API.create_mesh(V, T)
flywheel_position = ring_gear.position - face_width * V3.k()
flywheel_orientation = Q.Rx(np.pi / 2)
flywheel_name = API.add_object(engine, 'flywheel', flywheel_mesh, flywheel_position, flywheel_orientation)

API.add_hinge(engine, ring_name, flywheel_name, ring_gear.position, V3.k())

API.add_hinge(engine, flywheel_name, sun_name, flywheel_position, V3.k())
for planet_name, planet_gear in zip(planet_names, planet_gears):
    API.add_hinge(engine, ring_name, planet_name, planet_gear.position - face_width * V3.k(), V3.k())

engine.params.driver_angular_velocity = (np.pi / 6) * V3.k()

sun_velocities = []
planet_velocities = []
ring_velocities = []
def callback(engine: TYPES.Engine):
    sun_velocities.append(API.get_spin(engine, sun_name))
    planet_velocities.append([API.get_spin(engine, name) for name in planet_names])
    ring_velocities.append(API.get_spin(engine, ring_name))

app.run(200, callback=callback)

sun_velocities = np.array(sun_velocities)
planet_velocities = np.array(planet_velocities).reshape(-1, 3, 3)
ring_velocities = np.array(ring_velocities)

INFO:gear_app.GearApp:Initializing GearApp
INFO:add_object:Adding object sun_gear
INFO:create_shape:Grid cells: [ 393 2588 2591]
INFO:create_shape:Clipped grid cells: [128 128 128]
INFO:add_object:Adding object planet_gear_0
INFO:create_shape:Grid cells: [ 490 1857 1849]
INFO:create_shape:Clipped grid cells: [128 128 128]
INFO:add_object:Adding object planet_gear_1
INFO:create_shape:Grid cells: [ 490 1857 1849]
INFO:create_shape:Clipped grid cells: [128 128 128]
INFO:add_object:Adding object planet_gear_2
INFO:create_shape:Grid cells: [ 490 1857 1849]
INFO:create_shape:Clipped grid cells: [128 128 128]


KeyboardInterrupt: 

In [ ]:
planet_speeds = np.linalg.norm(planet_velocities, axis=1)

plt.figure(figsize=(6, 4), dpi=300)
for i in range(len(planet_names)):
    plt.plot(planet_speeds[:, i], label=f'Planet {i}')

plt.title(f'Planet Speeds (SDF Grid Size: {sdf_resolution}x{sdf_resolution}x{sdf_resolution})')
plt.xlabel('Steps')
plt.ylabel('Angular Speed')
plt.legend()
plt.savefig(f'planet_speeds_{sdf_resolution}.png')
plt.show()

NameError: name 'planet_velocities' is not defined